# BERTopic Analysis of Sustainability Reports

## 1. Introduction

This notebook applies BERTopic, a transformer-based topic modelling technique, to the sustainability reports collected from the selected companies.

Unlike traditional frequency analysis, BERTopic combines transformer embeddings with clustering techniques to automatically identify semantically coherent topics discussed throughout the sustainability reports. This approach provides a deeper understanding of the dominant sustainability themes communicated by companies.

The objectives of this notebook are to:

- Identify the principal sustainability topics contained in the reports.
- Extract the representative keywords associated with each topic.
- Examine the prevalence of the identified topics.
- Visualize the topic structure using BERTopic.
- Support the interpretation of sustainability communication through modern topic modelling techniques.

## 2. Import Required Libraries

The following libraries are required to perform transformer-based topic modelling using BERTopic.

In [1]:
# Install BERTopic (run only once)

!pip install bertopic
!pip install langdetect

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 55.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=809a23fa4252fedf385a7f3f0babcf562c16b6af8ee016083d0e4b1e7dfc7e18
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [2]:
import pandas as pd
import numpy as np

from bertopic import BERTopic

from sentence_transformers import SentenceTransformer

import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings("ignore")

# 3. Load the ESG Sentence Corpus

The BERTopic analysis is performed using the cleaned sentences extracted from the sustainability reports during the ESG classification stage. Each cleaned sentence is treated as an individual document, allowing BERTopic to discover detailed sustainability themes across the complete corpus.

Unlike the previous approach that treated each sustainability report as a single document, sentence-level topic modelling enables the identification of more specific Environmental, Social, Governance, and operational themes.

In [3]:
import zipfile

zip_path = "ESG_Sentence_Classification.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall()

print("ZIP extracted successfully.")

ZIP extracted successfully.


In [4]:
import os

os.listdir()

['.config',
 'ESG_Sentence_Classification.zip',
 'ESG_Sentence_Classification.xlsx',
 'sample_data']

In [5]:
documents_df = pd.read_excel(
    "ESG_Sentence_Classification.xlsx"
)

documents_df.head()

,Report,Sentence_ID,Original_Sentence,Cleaned_Sentence,ESG_Category,ESG_Score
0,Puratos-2025-Sustainability-GRI-Report,1,Better Life Planet Better Health Better 2025 S...,better life planet better health better 2025 s...,Environmental,1
1,Puratos-2025-Sustainability-GRI-Report,2,"We serve artisans, retailers, industrial and f...",we serve artisans retailers industrial and foo...,Governance,0
2,Puratos-2025-Sustainability-GRI-Report,3,"Our headquarters are located in Belgium, where...",our headquarters are located in belgium where ...,Governance,0
3,Puratos-2025-Sustainability-GRI-Report,4,"At Puratos, we believe that food has extraordi...",at puratos we believe that food has extraordin...,Governance,0
4,Puratos-2025-Sustainability-GRI-Report,5,We do not take such a responsibility lightly.,we do not take such a responsibility lightly,Governance,0


In [6]:
documents_df.columns.tolist()

['Report',
 'Sentence_ID',
 'Original_Sentence',
 'Cleaned_Sentence',
 'ESG_Category',
 'ESG_Score']

# 4. Create the BERTopic Documents

BERTopic requires a collection of textual documents as input. In this analysis, each cleaned sentence extracted from the sustainability reports is considered an individual document.

Using sentence-level documents enables BERTopic to identify more specific sustainability themes than would be possible when treating each report as a single document. This approach allows the model to capture fine-grained Environmental, Social, Governance, and operational topics discussed throughout the reports.

In [28]:
# Create the list of documents for BERTopic

documents = (
    documents_df["Cleaned_Sentence"]
    .dropna()
    .astype(str)
    .tolist()
)

import re

def clean_sentence(text):

    text = text.lower()

    # Remove years
    text = re.sub(r"\b20\d{2}\b", " ", text)

    # Remove standalone numbers
    text = re.sub(r"\b\d+\b", " ", text)

    # Remove GRI codes like 305-1, 403-2
    text = re.sub(r"\b\d+\s*[-.]\s*\d+\b", " ", text)

    # Remove punctuation
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

documents_df["Cleaned_Sentence"] = (
    documents_df["Cleaned_Sentence"]
    .astype(str)
    .apply(clean_sentence)
)

# Remove very short sentences
documents_df = documents_df[
    documents_df["Cleaned_Sentence"].str.split().str.len() >= 8
].reset_index(drop=True)

documents = documents_df["Cleaned_Sentence"].tolist()

print(len(documents))

26394


# 5. Generate Sentence Embeddings


To enable semantic topic modelling, each cleaned sustainability report sentence is converted into a dense numerical vector (embedding) using the pretrained **Sentence-BERT** model **all-MiniLM-L6-v2**.

Sentence embeddings capture the contextual meaning of text rather than relying solely on word frequency, allowing BERTopic to group semantically similar sentences even when different vocabulary is used.

The pretrained **all-MiniLM-L6-v2** model was selected because it provides an efficient balance between computational performance and semantic representation quality for English-language text.

**Reference**

Reimers, N., & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. Proceedings of EMNLP-IJCNLP.

In [29]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# Load Sentence-BERT model
# ------------------------------------------------------------

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ------------------------------------------------------------
# Build BERTopic model
# ------------------------------------------------------------

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="english",
    min_topic_size=100,
    calculate_probabilities=False,
    verbose=True
)

# ------------------------------------------------------------
# Fit BERTopic directly on the documents
# ------------------------------------------------------------

topics, probs = topic_model.fit_transform(documents)

print("BERTopic model created successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-07-29 11:07:00,896 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/825 [00:00<?, ?it/s]

2026-07-29 11:07:17,230 - BERTopic - Embedding - Completed ✓
2026-07-29 11:07:17,231 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-29 11:07:35,491 - BERTopic - Dimensionality - Completed ✓
2026-07-29 11:07:35,493 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-29 11:07:43,004 - BERTopic - Cluster - Completed ✓
2026-07-29 11:07:43,014 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-29 11:07:43,489 - BERTopic - Representation - Completed ✓


BERTopic model created successfully!


### Interpretation

Each sentence has been transformed into a contextual embedding that captures its semantic meaning. These embeddings provide the numerical representation required by BERTopic to identify semantically related sustainability topics across the report corpus.

# 6. BERTopic  Modelling

BERTopic is applied to the processed sustainability report sentences using contextual sentence embeddings generated by Sentence-BERT.

The model is configured with `language="english"` because the analysed sustainability reports are written exclusively in English. Consequently, topic representations are generated directly from the English corpus without any translation or multilingual preprocessing.

The parameter `min_topic_size=100` was selected to reduce the generation of very small or noisy topics and to ensure that the extracted topics represent recurring sustainability themes shared across multiple report sentences.

BERTopic combines transformer-based sentence embeddings with density-based clustering and class-based TF-IDF (c-TF-IDF) to automatically identify coherent sustainability topics without requiring predefined categories.

In [30]:
import nltk

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [31]:
print("Number of English sentences:", len(documents))

Number of English sentences: 26394


In [34]:
# ==========================================================
# BERTopic - ESG Topic Modelling (FINAL VERSION)
# ==========================================================

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from langdetect import detect
import numpy as np

# ==========================================================
# Keep only English sentences
# ==========================================================

def is_english(text):
    try:
        return detect(str(text)) == "en"
    except:
        return False

documents_df = documents_df[
    documents_df["Cleaned_Sentence"].apply(is_english)
].reset_index(drop=True)

documents = documents_df["Cleaned_Sentence"].tolist()

print(f"Number of English sentences: {len(documents)}")

# ==========================================================
# Stopwords
# ==========================================================

english_stopwords = stopwords.words("english")

custom_stopwords = {

    # Reporting
    "company","companies","group","business",
    "report","reports","reporting",
    "annual","year","years",
    "page","pages",
    "chapter","section",
    "appendix",
    "table","tables",
    "figure","figures",
    "statement","information",
    "management",
    "including","include","included",
    "according","following","across",
    "required",

    # Generic words
    "term","terms",
    "main",
    "number","numbers",
    "using","used","use",
    "provide","provides","provided",
    "support","supports","supported",
    "improve","improving",
    "continue","continues","continued",
    "make","made",
    "new",

    # ESG Framework
    "gri",
    "framework",
    "standard","standards",
    "indicator","indicators",
    "disclosure","disclosures",
    "reference","references",

    # Generic ESG
    "sustainability",
    "sustainable",
    "esg",

    # Strategy
    "strategy",
    "vision",
    "journey",
    "integrated",
    "impact",
    "impacts",

    # Governance
    "risk",
    "risks",

    # Years
    "2022","2023","2024","2025","2026",

    # Financial
    "million",
    "asset","assets",
    "cash",
    "financial",
    "income",
    "interest",
    "eur",
    "tax",
    "notes",

    # Company names
    "nestle",
    "danone",
    "jealsa",
    "froneri",
    "cargill",
    "puratos",
    "sovena",
    "borges",
    "azucarera",

    # Products
    "olive",
    "oil",
    "ice",
    "cream",
    "cocoa",
    "fish",
    "tuna",
    "seafood",
    "bakery",
    "palm",
    "rspo",

    # CSR
    "csr",
    "committed",
    "international",

    # Environment
    "environment",
    "environmental"
}
custom_stopwords.update({
    # Generic report words
    "non", "good", "important", "making", "make", "made",
    "together", "two", "order", "document", "account",
    "process", "11", "10", "12", "13", "14", "15"


    # Generic report words
    "project",
    "projects",
    "process",
    "processes",
    "people",
    "important",
    "include",
    "includes",
    "including",
    "provide",
    "provides",
    "provided",
    "using",
    "used",
    "continued",
    "continue",
    "main",
    "annual",
    "integrated",
    "statement",
    "information",
    "page",
    "pages",
    "chapter",
    "section",
    "table",
    "tables",
    "figure",
    "figures",
    "appendix",

# Numbers written as words
"one",
"two",
"three",
})


all_stopwords = list(set(english_stopwords + list(custom_stopwords)))

# ==========================================================
# CountVectorizer
# ==========================================================

vectorizer_model = CountVectorizer(
    stop_words=all_stopwords,
    ngram_range=(1, 2),
    min_df=10,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
)
# ==========================================================
# Sentence Embeddings (GPU)
# ==========================================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda"
)

embeddings = embedding_model.encode(
    documents,
    batch_size=512,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Save embeddings (recommended)

np.save("embeddings.npy", embeddings)

# ==========================================================
# BERTopic
# ==========================================================

representation_model = KeyBERTInspired()

topic_model = BERTopic(
    embedding_model=embedding_model,      # <-- CHANGE THIS
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    language="english",
    min_topic_size=60,
    nr_topics=10,
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model.fit_transform(documents)
# ==========================================================
# Topic Information
# ==========================================================

topic_info = topic_model.get_topic_info()

topic_info = topic_info[topic_info["Topic"] != -1]

display(topic_info)

print(f"Final number of topics: {len(topic_info)}")

topic_model.visualize_barchart(
    top_n_topics=10,
    n_words=10
)

topic_model.visualize_topics()

topic_model.visualize_hierarchy()

Number of English sentences: 26368


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/52 [00:00<?, ?it/s]

2026-07-29 12:02:22,659 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/824 [00:00<?, ?it/s]

2026-07-29 12:02:36,919 - BERTopic - Embedding - Completed ✓
2026-07-29 12:02:36,920 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-29 12:02:58,621 - BERTopic - Dimensionality - Completed ✓
2026-07-29 12:02:58,624 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-29 12:03:04,125 - BERTopic - Cluster - Completed ✓
2026-07-29 12:03:04,126 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-07-29 12:03:04,940 - BERTopic - Representation - Completed ✓
2026-07-29 12:03:04,941 - BERTopic - Topic reduction - Reducing number of topics
2026-07-29 12:03:04,982 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-29 12:03:06,150 - BERTopic - Representation - Completed ✓
2026-07-29 12:03:06,158 - BERTopic - Topic reduction - Reduced number of topics from 72 to 10


,Topic,Count,Name,Representation,Representative_Docs
1,0,4330,0_supply chain_compliance_consumers_consumer,"[supply chain, compliance, consumers, consumer...",[impact report overview strategy land and wate...
2,1,1885,1_safety_compliance_regulations_employees,"[safety, compliance, regulations, employees, e...",[the audit documents shared with nestl in sede...
3,2,1804,2_stakeholders_governance_responsibility_initi...,"[stakeholders, governance, responsibility, ini...",[main numbers main numbers letter from the cha...
4,3,1721,3_scope_assess_footprint_measures,"[scope, assess, footprint, measures, productio...",[emissions change vs baseline climate targets ...
5,4,1452,4_compliance_investment_market_resources,"[compliance, investment, market, resources, go...",[non financial statement table of contents abo...
6,5,968,5_training_education_developing_workforce,"[training, education, developing, workforce, d...",[training clear communication and teamwork mak...
7,6,964,6_rights_principles_compliance_united,"[rights, principles, compliance, united, gover...",[the policy formalises bolton s commitment to ...
8,7,866,7_waste_footprint_production_facilities,"[waste, footprint, production, facilities, red...",[in accordance with the recommendations of the...
9,8,654,8_water_reducing_reduce_reduction,"[water, reducing, reduce, reduction, waste, fa...",[l seafman ecuador water from retorts is filte...


Final number of topics: 9


In [33]:
import bertopic
print(bertopic.__version__)

0.17.4


### Interpretation

The BERTopic model automatically identifies latent sustainability themes by clustering semantically similar sentences extracted from the sustainability reports. Sentence representations were generated using the **all-MiniLM-L6-v2 SentenceTransformer**, while BERTopic combined contextual embeddings, dimensionality reduction (UMAP), density-based clustering (HDBSCAN), and class-based TF-IDF (c-TF-IDF) to derive representative keywords for each topic.

Prior to topic modelling, the corpus was restricted to **English-language sentences** and preprocessed by removing English stopwords together with additional domain-specific reporting terms (e.g., GRI references, generic reporting vocabulary, financial reporting terms, and company-specific artefacts). This preprocessing improves topic coherence by reducing non-informative words while preserving sustainability-related terminology.

The resulting topics provide an unsupervised thematic overview of the sustainability reports and complement the descriptive NLP analysis and ESG classification presented in the previous notebooks.

# 7. Topic Information

This section presents an overview of the topics automatically identified by BERTopic from the English sustainability report corpus.

Each topic is assigned a unique identifier together with the number of sentences associated with that topic and a set of representative keywords extracted using class-based TF-IDF (c-TF-IDF). The representative keywords summarize the dominant sustainability concepts discussed within each topic.

To improve the interpretability of the discovered topics, English stopwords and generic reporting terms were removed during preprocessing. Consequently, the extracted keywords focus on meaningful sustainability-related concepts rather than common document terms.

In [35]:
# ==========================================================
# Topic Information
# ==========================================================

topic_info = topic_model.get_topic_info()

# Remove the outlier topic (-1)
topic_info = topic_info[topic_info["Topic"] != -1]

display(topic_info)

print(f"Number of final topics: {len(topic_info)}")

,Topic,Count,Name,Representation,Representative_Docs
1,0,4330,0_supply chain_compliance_consumers_consumer,"[supply chain, compliance, consumers, consumer...",[impact report overview strategy land and wate...
2,1,1885,1_safety_compliance_regulations_employees,"[safety, compliance, regulations, employees, e...",[the audit documents shared with nestl in sede...
3,2,1804,2_stakeholders_governance_responsibility_initi...,"[stakeholders, governance, responsibility, ini...",[main numbers main numbers letter from the cha...
4,3,1721,3_scope_assess_footprint_measures,"[scope, assess, footprint, measures, productio...",[emissions change vs baseline climate targets ...
5,4,1452,4_compliance_investment_market_resources,"[compliance, investment, market, resources, go...",[non financial statement table of contents abo...
6,5,968,5_training_education_developing_workforce,"[training, education, developing, workforce, d...",[training clear communication and teamwork mak...
7,6,964,6_rights_principles_compliance_united,"[rights, principles, compliance, united, gover...",[the policy formalises bolton s commitment to ...
8,7,866,7_waste_footprint_production_facilities,"[waste, footprint, production, facilities, red...",[in accordance with the recommendations of the...
9,8,654,8_water_reducing_reduce_reduction,"[water, reducing, reduce, reduction, waste, fa...",[l seafman ecuador water from retorts is filte...


Number of final topics: 9


# 8. Reduced Topic Information

The initial BERTopic model identified a relatively large number of fine-grained topics. To facilitate interpretation and discussion, BERTopic's topic reduction procedure was applied to merge semantically similar topics into ten broader sustainability themes.

The reduced topic structure provides a concise overview of the principal sustainability themes discussed across the analysed reports while preserving the semantic relationships identified during topic modelling.

In [36]:
# ==========================================================
# Reduce Topics (ONLY if more than 10 topics)
# ==========================================================

topic_info = topic_model.get_topic_info()

# Remove outlier topic (-1)
topic_info = topic_info[topic_info["Topic"] != -1]

# Only reduce if there are more than 10 topics
if len(topic_info) > 10:

    topic_model = topic_model.reduce_topics(
        documents,
        nr_topics=10
    )

    topic_info = topic_model.get_topic_info()
    topic_info = topic_info[topic_info["Topic"] != -1]

display(topic_info)

,Topic,Count,Name,Representation,Representative_Docs
1,0,4330,0_supply chain_compliance_consumers_consumer,"[supply chain, compliance, consumers, consumer...",[impact report overview strategy land and wate...
2,1,1885,1_safety_compliance_regulations_employees,"[safety, compliance, regulations, employees, e...",[the audit documents shared with nestl in sede...
3,2,1804,2_stakeholders_governance_responsibility_initi...,"[stakeholders, governance, responsibility, ini...",[main numbers main numbers letter from the cha...
4,3,1721,3_scope_assess_footprint_measures,"[scope, assess, footprint, measures, productio...",[emissions change vs baseline climate targets ...
5,4,1452,4_compliance_investment_market_resources,"[compliance, investment, market, resources, go...",[non financial statement table of contents abo...
6,5,968,5_training_education_developing_workforce,"[training, education, developing, workforce, d...",[training clear communication and teamwork mak...
7,6,964,6_rights_principles_compliance_united,"[rights, principles, compliance, united, gover...",[the policy formalises bolton s commitment to ...
8,7,866,7_waste_footprint_production_facilities,"[waste, footprint, production, facilities, red...",[in accordance with the recommendations of the...
9,8,654,8_water_reducing_reduce_reduction,"[water, reducing, reduce, reduction, waste, fa...",[l seafman ecuador water from retorts is filte...


### Interpretation

Following topic reduction, BERTopic grouped semantically similar topics into ten broader sustainability themes. This reduction improves interpretability by minimizing overlapping topics and emphasizing the principal environmental, social, and governance issues discussed across the sustainability reports.

The resulting topics provide a concise thematic summary suitable for subsequent visualisation and discussion.

# 9. Intertopic Distance Map

The intertopic distance map provides a visual representation of the semantic relationships among the ten reduced sustainability topics identified from the English sustainability report corpus.

Each circle represents one topic, while the distance between circles reflects the semantic similarity between topics based on their contextual sentence embeddings.

In [37]:
topic_model.visualize_topics()

### Interpretation

The intertopic distance map provides a visual overview of the semantic relationships among the sustainability topics identified from the analysed sustainability reports.

Topics located close together discuss semantically related sustainability issues, whereas topics positioned farther apart represent more distinct themes. Larger circles indicate topics containing a greater number of report sentences.

Overall, the visualization illustrates how the identified sustainability themes are distributed across the complete corpus of sustainability reports and highlights the degree of similarity between the discovered topics.

## 10.Topic Keyword Representation

This section presents the most representative keywords associated with each of the ten reduced sustainability topics identified by BERTopic.

The keywords are extracted using class-based Term Frequency–Inverse Document Frequency (c-TF-IDF), which identifies the terms that best characterize each topic relative to the remaining corpus. Displaying only the reduced topics provides a clearer overview of the principal sustainability themes discussed across the English sustainability reports.

In [38]:
# Display the top keywords of each reduced topic

for topic in topic_model.get_topics().keys():

    if topic != -1:

        print(f"\nTopic {topic}")

        print(topic_model.get_topic(topic))


Topic 0
[('supply chain', np.float32(0.37999013)), ('compliance', np.float32(0.37709412)), ('consumers', np.float32(0.35344735)), ('consumer', np.float32(0.30217916)), ('foods', np.float32(0.29760486)), ('organizations', np.float32(0.296545)), ('partnership', np.float32(0.29335558)), ('suppliers', np.float32(0.28714553)), ('supply', np.float32(0.28092328)), ('industry', np.float32(0.2759183))]

Topic 1
[('safety', np.float32(0.50427765)), ('compliance', np.float32(0.47230458)), ('regulations', np.float32(0.42239562)), ('employees', np.float32(0.3839435)), ('ensuring', np.float32(0.3832023)), ('policies', np.float32(0.37023464)), ('prevention', np.float32(0.36913255)), ('nestl', np.float32(0.34561768)), ('practices', np.float32(0.33606392)), ('policy', np.float32(0.32811555))]

Topic 2
[('stakeholders', np.float32(0.52911055)), ('governance', np.float32(0.39429045)), ('responsibility', np.float32(0.34381497)), ('initiatives', np.float32(0.32294378)), ('responsible', np.float32(0.320993

# 11. Topic Distribution

This section presents the distribution of the identified sustainability topics. The bar chart displays the relative frequency of each topic within the English sustainability reports, providing an overview of the most prevalent sustainability themes identified by BERTopic.

In [39]:
topic_model.visualize_barchart(
    top_n_topics=9,
    n_words=10
)

##Interpretation

Topic 0 – Compliance and Supply Chain
This topic focuses on compliance practices, supplier management, and relationships with consumers across the supply chain.

Topic 1 – Corporate Policies and Employees
This topic reflects corporate policies, employee responsibilities, and organisational compliance.

Topic 2 – Corporate Governance and ESG Initiatives
This topic represents governance practices, sustainability initiatives, and organisational objectives.

Topic 3 – ESG Performance Measurement
This topic relates to assessing sustainability performance through metrics, measures, and monitoring activities.

Topic 4 – Resource and Investment Management
This topic focuses on resource allocation, investment decisions, and internal organisational management.

Topic 5 – Employee Development and Leadership
This topic highlights workforce training, employee development, and leadership initiatives
Topic 6 – Corporate Principles and Leadership
This topic represents organisational values, leadership principles, and strategic objectives.

Topic 7 – Environmental Footprint Management
This topic focuses on reducing environmental impacts through improvements in facilities and operations.

Topic 8 – Operational Efficiency and Resource Reduction
This topic reflects initiatives aimed at improving operational efficiency and reducing resource consumption.


## Overall Summary of BERTopic Results

The BERTopic analysis identified nine interpretable sustainability themes across the analysed reports. The topics cover governance, compliance, employee development, environmental management, operational efficiency, and resource management. Although some overlap exists between governance-related topics, this is expected because ESG issues are frequently interconnected within sustainability reports. Overall, the results demonstrate that companies address sustainability through a combination of environmental, social, and governance initiatives, providing a comprehensive thematic overview of corporate sustainability reporting.